# Crack-tip fields: examples and validation

This notebook demonstrates and validates the crack-tip fields implemented in
`crack_tip_fields.py` for three cases:

1. **Isotropic** linear-elastic behaviour (classical LEFM near-tip solution)
2. **Anisotropic** linear-elastic behaviour in a homogeneous single crystal (Stroh formalism)
3. **Interface cracks** between two anisotropic grains with different orientation and/or
   material (oscillating singularity)

All physics is implemented in the module `crack_tip_fields.py`; this notebook contains
no duplicate implementations and only calls the module. The interactive Abaqus
boundary-condition app (`abaqus_crack_tip_bc_app.ipynb`) uses the same module.

**Units:** mm, N, s, MPa (so $K$ has the unit MPa$\cdot\sqrt{\text{mm}}$).

## References

- C. Hwu, *Anisotropic Elastic Plates*, Springer, 2010 (Ch. 3, 5, 7, 11)
- P. Vannucci, *Anisotropic Elasticity*, Springer, 2018 (Ch. 3.10)
- F. Brunner, P. Andric, F. Maresca, *An atomistic K-test framework for general grain
  boundaries and triclinic single crystals*, Modelling Simul. Mater. Sci. Eng. 33 (2025)
  035004, https://doi.org/10.1088/1361-651X/adba03

## Structure

All input parameters (stress intensity factors, crack length, materials and grain
orientations) are set centrally in section **3 "Configuration"**. The following sections
use these values. New materials can be added to the dictionary `MATERIALS` in
`crack_tip_fields.py`.

> **Stresses:** For all three cases the stresses are available in closed form. For the
> anisotropic and interface cases they are derived from the Stroh stress function
> (see the accompanying paper) and verified in section 11 against finite differences of the
> displacement field.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))  # crack_tip_fields.py in ../src

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation

from crack_tip_fields import (
    MATERIALS, print_engineering_constants, rotate_stiffness_euler, isotropic_stiffness,
    IsotropicCrackModel, AnisotropicCrackModel, InterfaceCrackModel,
    stress_field_isoElast, von_mises_3d, hydro_stress_3d,
    displacement_field_anisoElast, displacement_field_anisoElast_GB,
    strain_stress_fd, compute_field_grid, plot_heatmap,
    compute_eff_compliance, compute_critical_K, calculate_E, calculate_Eeff,
    EulerAnglesDegFromDirection, direction_from_EulerAnglesDeg, vector_to_miller,
)

np.set_printoptions(precision=4, suppress=True)

## 1. Material database

Each entry of `MATERIALS` contains the stiffness matrix $\mathbf{C}$ in Voigt notation
(6×6, MPa) in the **crystal coordinate system**, with the convention

$$[11,\,22,\,33,\,23,\,13,\,12]$$

New materials can be added by one more dictionary entry in `crack_tip_fields.py`
(cubic via `cubic_stiffness`, isotropic via `isotropic_stiffness`, or directly as a full
6×6 matrix for lower symmetries, e.g. orthotropic or triclinic).

In [ ]:
for _name, _mat in MATERIALS.items():
    print_engineering_constants(_mat["C"], label=_name)

## 2. Crystal orientation and rotation of the stiffness matrix

The stiffness matrices in `MATERIALS` are defined in the **crystal coordinate system**.
For the crack-tip fields the stiffness matrix is rotated into the **global coordinate
system** in which the crack or interface has the normal $[0,1,0]$. The rotation uses
Bunge Euler angles ($z$-$x$-$z$ convention), `rotate_stiffness_euler`.

## 3. Configuration

**Adjust all inputs here:** stress intensity factors, crack length, evaluation region, and
material and orientation of the two grains. All following sections use these values.

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Stress intensity factors [MPa * sqrt(mm)]
K_I, K_II, K_III = 5. * np.sqrt(1000.), 0.0, 0.0
k_vec = np.array([K_II, K_I, K_III])   # order as expected by the Stroh formalism

# Crack half-length [mm]
a = 0.05

# Evaluation region around the crack tip (rule of thumb: 5-10 % of the crack length = K-dominated zone)
x_K = 0.1 * a

# --- Grain 1 (above the interface, x2 > 0) ---
material_1_name = "Fe (cubic, example)"
phi1_1, Phi_1, phi2_1 = 27., 0., 0.     # Euler angles [deg], Bunge convention z-x-z

# --- Grain 2 (below the interface, x2 < 0) ---
material_2_name = "Fe (cubic, example)"
phi1_2, Phi_2, phi2_2 = 0., 0., 0.

# ------------------------------------------------------------
C1 = MATERIALS[material_1_name]["C"]
C2 = MATERIALS[material_2_name]["C"]

C_rot1 = rotate_stiffness_euler(C1, phi1_1, Phi_1, phi2_1)
C_rot2 = rotate_stiffness_euler(C2, phi1_2, Phi_2, phi2_2)

print(f"K_I = {K_I:.3f}, K_II = {K_II:.3f}, K_III = {K_III:.3f}  [MPa*sqrt(mm)]")
print(f"a = {a} mm,  x_K (evaluation region) = {x_K} mm\n")

print_engineering_constants(C1, f"Grain 1 - {material_1_name} (crystal system)")
print_engineering_constants(C_rot1, f"Grain 1 - {material_1_name} (global system, phi1={phi1_1} Phi={Phi_1} phi2={phi2_1})")
print()
print_engineering_constants(C2, f"Grain 2 - {material_2_name} (crystal system)")
print_engineering_constants(C_rot2, f"Grain 2 - {material_2_name} (global system, phi1={phi1_2} Phi={Phi_2} phi2={phi2_2})")

## 4. Crack-tip field: isotropic linear-elastic

Classical LEFM near-tip solution (Williams series, first term) for modes I/II/III,
plane strain (`IsotropicCrackModel`, `stress_field_isoElast`).

In [ ]:
# Example material for the isotropic case (free choice, independent of grains 1/2 above)
iso_material_name = "Steel (isotropic)"
iso_model = IsotropicCrackModel(iso_material_name)   # plane strain; plane="stress" for plane stress
E_iso, nu_iso, G_iso, kappa_iso = iso_model.E, iso_model.nu, iso_model.G, iso_model.kappa

print(f"{iso_material_name}: E = {E_iso:.0f} MPa, G = {G_iso:.0f} MPa, nu = {nu_iso:.3f}")

# Displacement and stress field on a grid around the crack tip
x_vals = np.linspace(-x_K, x_K, 300)
y_vals = np.linspace(-x_K, x_K, 300)
X1, X2 = np.meshgrid(x_vals, y_vals)

U1, U2, U3 = iso_model.displacement(X1, X2, K_I, K_II, K_III)
plot_heatmap(U1, X1, X2, "Displacement u1 (isotropic)", "u1 [mm]")
plot_heatmap(U2, X1, X2, "Displacement u2 (isotropic)", "u2 [mm]")
plot_heatmap(U3, X1, X2, "Displacement u3 (mode III, isotropic)", "u3 [mm]")

sxx, syy, szz, sxy, sxz, syz = stress_field_isoElast(K_I, K_II, K_III, X1, X2, nu_plane_strain=nu_iso)  # szz = nu*(sxx+syy)
von_mises = von_mises_3d(sxx, syy, szz, sxy, sxz, syz)
hydro = hydro_stress_3d(sxx, syy, szz, sxy, sxz, syz)

# The 1/sqrt(r) singularity at the crack tip would otherwise dominate the colour scale -> clip
sigma_cap = 2500.  # MPa
sxx_c = np.clip(sxx, -sigma_cap, sigma_cap)
syy_c = np.clip(syy, -sigma_cap, sigma_cap)
sxy_c = np.clip(sxy, -sigma_cap, sigma_cap)
von_mises_c = np.clip(von_mises, None, sigma_cap)
hydro_c = np.clip(hydro, -sigma_cap, sigma_cap)

plot_heatmap(sxx_c, X1, X2, "Normal stress sigma_xx (isotropic)", "sigma_xx [MPa]")
plot_heatmap(syy_c, X1, X2, "Normal stress sigma_yy (isotropic)", "sigma_yy [MPa]")
plot_heatmap(sxy_c, X1, X2, "Shear stress sigma_xy (isotropic)", "sigma_xy [MPa]")
plot_heatmap(von_mises_c, X1, X2, "von Mises equivalent stress (isotropic)", "sigma_v [MPa]")
plot_heatmap(hydro_c, X1, X2, "Hydrostatic stress (isotropic)", "sigma_h [MPa]")

In [ ]:
# Crack opening / sliding displacement along x1 < 0
x1_vals = np.linspace(-x_K, -1e-5, 100)
x2_top, x2_bot = 1e-10, -1e-10

u_top = [iso_model.displacement(x1i, x2_top, K_I, K_II, K_III)[1] for x1i in x1_vals]
u_bot = [iso_model.displacement(x1i, x2_bot, K_I, K_II, K_III)[1] for x1i in x1_vals]
cod = np.array(u_top) - np.array(u_bot)

plt.figure(figsize=(6, 4))
plt.plot(x1_vals, u_top, "k:", label="upper crack face")
plt.plot(x1_vals, u_bot, "k--", label="lower crack face")
plt.plot(x1_vals, cod, "r-", label="CTOD (opening)")
plt.xlabel("x1 [mm]")
plt.ylabel("Crack-face displacement [mm]")
plt.title("Crack opening displacement (isotropic)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## 5. Crack-tip field: anisotropic, homogeneous single crystal (Stroh formalism)

The Stroh formalism solves the plane (or generalised plane) boundary-value problem for
anisotropic elastic behaviour via an eigenvalue problem of the 6×6 system matrix
$\mathbf{N}$, built from the blocks $\mathbf{Q}, \mathbf{R}, \mathbf{T}$ of the stiffness
matrix (Hwu 2010, Ch. 3 and 11).

A homogeneous single crystal is the special case of the bimaterial solution with
identical grains: $\mathbf{W}=\mathbf{0}$, $\hat{\mathbf S}=\mathbf{0}$, i.e. no oscillation
(`AnisotropicCrackModel`).

**Stresses:** closed-form, `AnisotropicCrackModel.stress` (including $\sigma_{33}$ from
$\varepsilon_{33}=0$). Section 11 verifies them against finite differences
($\varepsilon_{ij}=\tfrac12(\partial u_i/\partial x_j+\partial u_j/\partial x_i)$,
$\varepsilon_{33}=0$, Hooke's law).

In [ ]:
# Homogeneous single crystal = grain 1 on both sides -> W = 0, S_hat = 0 (no oscillation)
single = AnisotropicCrackModel(material_1_name, phi1_1, Phi_1, phi2_1)

print("p  =", single.p)
print("S_hat (0 for a single crystal) =\n", single.S_hat)

x1_pt, x2_pt = -0.01, 1e-10
u1p, u2p, u3p = single.displacement(x1_pt, x2_pt, K_I, K_II, K_III, a)
print(f"\nu(x1={x1_pt}, x2={x2_pt}) = ({u1p:.6e}, {u2p:.6e}, {u3p:.6e}) mm")

n_grid = 60   # grid resolution; higher values increase the run time (see compute_field_grid)
x_vals = np.linspace(-x_K, x_K, n_grid)
y_vals = np.linspace(-x_K, x_K, n_grid)
X1, X2 = np.meshgrid(x_vals, y_vals)

U1, U2, U3, SIG = compute_field_grid(
    lambda x, y: single.displacement(x, y, K_I, K_II, K_III, a), single.C_rot, x_vals, y_vals, a,
    stress_func=lambda x, y: single.stress(x, y, K_I, K_II, K_III, a))

plot_heatmap(U1, X1, X2, "Displacement u1 (anisotropic, single crystal)", "u1 [mm]")
plot_heatmap(U2, X1, X2, "Displacement u2 (anisotropic, single crystal)", "u2 [mm]")
plot_heatmap(U3, X1, X2, "Displacement u3 (anisotropic, single crystal)", "u3 [mm]")

sigma_cap = 2500.  # MPa
s11 = np.clip(SIG[:, :, 0], -sigma_cap, sigma_cap)
s22 = np.clip(SIG[:, :, 1], -sigma_cap, sigma_cap)
s12 = np.clip(SIG[:, :, 5], -sigma_cap, sigma_cap)

plot_heatmap(s11, X1, X2, "Normal stress sigma_11 (anisotropic, single crystal)", "sigma_11 [MPa]")
plot_heatmap(s22, X1, X2, "Normal stress sigma_22 (anisotropic, single crystal)", "sigma_22 [MPa]")
plot_heatmap(s12, X1, X2, "Shear stress sigma_12 (anisotropic, single crystal)", "sigma_12 [MPa]")

In [ ]:
# Crack opening displacement (CTOD) for the homogeneous anisotropic single crystal
x1_vals = np.linspace(-x_K, -1e-5, 100)
x2_top, x2_bot = 1e-10, -1e-10

u_top = [single.displacement(x1i, x2_top, K_I, K_II, K_III, a)[1] for x1i in x1_vals]
u_bot = [single.displacement(x1i, x2_bot, K_I, K_II, K_III, a)[1] for x1i in x1_vals]
cod = np.array(u_top) - np.array(u_bot)

plt.figure(figsize=(6, 4))
plt.plot(x1_vals, u_top, "k:", label="upper crack face")
plt.plot(x1_vals, u_bot, "k--", label="lower crack face")
plt.plot(x1_vals, cod, "r-", label="CTOD (opening)")
plt.xlabel("x1 [mm]")
plt.ylabel("Crack-face displacement [mm]")
plt.title("Crack opening displacement (anisotropic, single crystal)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## 6. Interface crack between two anisotropic grains

At the interface between two differently oriented or different grains an **oscillating
singularity** occurs, controlled by the bimaterial matrix
$\hat{\mathbf S}=\mathbf D^{-1}\mathbf W$ (Hwu 2010, Ch. 7). For identical material on both
sides (section 5) the oscillation vanishes ($\hat{\mathbf S}=0$).
`InterfaceCrackModel` evaluates grain 1 above and grain 2 below the interface.

In [ ]:
interface = InterfaceCrackModel(material_1_name, (phi1_1, Phi_1, phi2_1),
                                material_2_name, (phi1_2, Phi_2, phi2_2))
S_hat, D, W = interface.S_hat, interface.D, interface.W

print("S_hat (oscillation matrix) =\n", S_hat)

u_top, u_bot = displacement_field_anisoElast_GB(
    interface.p1, interface.A1, interface.B1, interface.p2, interface.A2, interface.B2,
    S_hat, x1_pt, x2_pt, k_vec, a)
print(f"\nu(grain 1, top)    at (x1={x1_pt}, x2={x2_pt}): {u_top}")
print(f"u(grain 2, bottom) at (x1={x1_pt}, x2={x2_pt}): {u_bot}")

In [ ]:
# Crack opening/sliding displacement at the interface (at a finite distance from the interface,
# since top/bot become identical for x2 -> 0)
x1_vals = np.linspace(-x_K, -1e-5, 100)
x2_val = 0.0005

u_top2, u_bot2, cod = [], [], []
u_top1, u_bot1, csd = [], [], []
for x1i in x1_vals:
    u_t, u_b = displacement_field_anisoElast_GB(
        interface.p1, interface.A1, interface.B1, interface.p2, interface.A2, interface.B2,
        S_hat, x1i, x2_val, k_vec, a)
    u_top2.append(u_t[1]); u_bot2.append(u_b[1]); cod.append(u_t[1] - u_b[1])
    u_top1.append(u_t[0]); u_bot1.append(u_b[0]); csd.append(u_t[0] - u_b[0])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(x1_vals, u_top2, "k--", label="top")
axes[0].plot(x1_vals, u_bot2, "k:", label="bottom")
axes[0].plot(x1_vals, cod, "r-", label="CTOD")
axes[0].set_xlabel("x1 [mm]"); axes[0].set_ylabel("u2 [mm]")
axes[0].set_title("Crack opening displacement (interface)")
axes[0].grid(True); axes[0].legend()

axes[1].plot(x1_vals, u_top1, "k--", label="top")
axes[1].plot(x1_vals, u_bot1, "k:", label="bottom")
axes[1].plot(x1_vals, csd, "r-", label="CTSD")
axes[1].set_xlabel("x1 [mm]"); axes[1].set_ylabel("u1 [mm]")
axes[1].set_title("Crack sliding displacement (interface)")
axes[1].grid(True); axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Displacement and stress fields above and below the interface
n_grid_gb = 50   # grid resolution per half plane (run time scales quadratically!)

x1_range = np.linspace(-x_K, x_K, n_grid_gb)
x2_top_range = np.linspace(1e-10, x_K, n_grid_gb // 2)
x2_bot_range = np.linspace(-x_K, -1e-10, n_grid_gb // 2)

U1t, U2t, U3t, SIGt = compute_field_grid(
    lambda x, y: interface.displacement(x, y, K_I, K_II, K_III, a, side="top"),
    interface.C_rot_top, x1_range, x2_top_range, a,
    stress_func=lambda x, y: interface.stress(x, y, K_I, K_II, K_III, a, side="top"))
U1b, U2b, U3b, SIGb = compute_field_grid(
    lambda x, y: interface.displacement(x, y, K_I, K_II, K_III, a, side="bot"),
    interface.C_rot_bot, x1_range, x2_bot_range, a,
    stress_func=lambda x, y: interface.stress(x, y, K_I, K_II, K_III, a, side="bot"))

X1t, X2t = np.meshgrid(x1_range, x2_top_range)
X1b, X2b = np.meshgrid(x1_range, x2_bot_range)

# Displacements: common colour scale for top + bottom
for name, Ut, Ub in [("u1", U1t, U1b), ("u2", U2t, U2b), ("u3", U3t, U3b)]:
    vmax = max(np.abs(Ut).max(), np.abs(Ub).max())
    plot_heatmap(Ut, X1t, X2t, f"Displacement {name} (grain 1, top)", f"{name} [mm]", vmin=-vmax, vmax=vmax)
    plot_heatmap(Ub, X1b, X2b, f"Displacement {name} (grain 2, bottom)", f"{name} [mm]", vmin=-vmax, vmax=vmax)

# Stresses (clipped because of the 1/sqrt(r) singularity), common scale top/bottom
sigma_cap = 2500.
for name, idx in [("sigma_11", 0), ("sigma_22", 1), ("sigma_12", 5)]:
    St = np.clip(SIGt[:, :, idx], -sigma_cap, sigma_cap)
    Sb = np.clip(SIGb[:, :, idx], -sigma_cap, sigma_cap)
    plot_heatmap(St, X1t, X2t, f"{name} (grain 1, top)", f"{name} [MPa]", vmin=-sigma_cap, vmax=sigma_cap)
    plot_heatmap(Sb, X1b, X2b, f"{name} (grain 2, bottom)", f"{name} [MPa]", vmin=-sigma_cap, vmax=sigma_cap)

## 7. Critical stress intensity factor (Griffith criterion)

Estimate of the critical $K_{Ic}$ for crack propagation along the interface from the
Griffith energy release rate, $G_c=\gamma_{s1}+\gamma_{s2}-\gamma_{gb}$, and the
effective compliance $\mathbf E = \mathbf D+\mathbf W\mathbf D^{-1}\mathbf W$
(Hwu 2010, Eq. 7.114): `compute_eff_compliance`, `compute_critical_K`.

In [ ]:
# Example values (following Brunner, Andric, Maresca, Table 6, for an Fe-Fe grain boundary)
gamma_s1 = 2.0 / 1000.   # [mJ/mm^2]
gamma_s2 = 2.0 / 1000.
gamma_gb = 0.3 / 1000.

E_eff = compute_eff_compliance(D, W)
E22 = E_eff[1, 1]
print("E (effective compliance) =\n", E_eff)
print(f"E22 = {E22:.6e} 1/MPa")

K_Ic = compute_critical_K(E22, gamma_s1, gamma_s2, gamma_gb)
print(f"\nK_Ic = {K_Ic:.3f} MPa*sqrt(mm)  =  {K_Ic / np.sqrt(1000.):.3f} MPa*sqrt(m)")

## 8. Bonus: directional Young's modulus (cubic symmetry)

Visualises the directional dependence of Young's modulus of grain 1 as a 3D surface and
histogram (valid for cubic symmetry only), `calculate_E`.

In [ ]:
C11_1, C12_1, C44_1 = C1[0, 0], C1[0, 1], C1[3, 3]
zener_A = 2 * C44_1 / (C11_1 - C12_1)
print(f"Grain 1 ({material_1_name}): C11={C11_1:.1f}, C12={C12_1:.1f}, C44={C44_1:.1f} MPa")
print(f"Zener anisotropy factor A = {zener_A:.4f}  (A=1: isotropic)")

# --- Surface plot ---
N_theta, N_phi = 90, 180
theta_vals = np.linspace(0, np.pi, N_theta)
phi_vals = np.linspace(0, 2 * np.pi, N_phi)
Theta, Phi = np.meshgrid(theta_vals, phi_vals)

X = np.sin(Theta) * np.cos(Phi)
Y = np.sin(Theta) * np.sin(Phi)
Z = np.cos(Theta)

E_dir = np.array([[calculate_E([X[i, j], Y[i, j], Z[i, j]], C11_1, C12_1, C44_1)
                    for j in range(X.shape[1])] for i in range(X.shape[0])])

norm = plt.Normalize(E_dir.min(), E_dir.max())
facecolors = plt.cm.viridis(norm(E_dir))

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X * E_dir, Y * E_dir, Z * E_dir, facecolors=facecolors, rstride=1, cstride=1, linewidth=0)
mappable = plt.cm.ScalarMappable(cmap='viridis', norm=norm)
mappable.set_array([])
fig.colorbar(mappable, ax=ax, shrink=0.5, aspect=10, label="Young's modulus [MPa]")
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_box_aspect([1, 1, 1])
ax.set_title(f"Directional Young's modulus: {material_1_name}")
plt.tight_layout()
plt.show()

# --- Histogram ---
weights = np.ones_like(E_dir.ravel()) / E_dir.size
plt.figure(figsize=(6, 4))
plt.hist(E_dir.ravel(), bins=30, weights=weights, color='skyblue', edgecolor='black', alpha=0.7)
plt.xlabel("Young's modulus [MPa]")
plt.ylabel('relative frequency')
plt.title(f"Distribution of the directional Young's modulus: {material_1_name}")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 9. Bonus: effective interface stiffness versus misorientation

Computes $4/E_{22}$ (a measure of the effective stiffness normal to the interface, cf.
section 7) for grain 1 in all orientations against the fixed grain 2 and shows the result
as a 3D scatter plot over the rotation axis (sample direction $z$), `calculate_Eeff`.

> ⚠️ **Computationally intensive:** a complete Stroh eigenvalue problem is solved for every
> combination of Euler angles. The default grid (step 20°) usually runs within a few
> minutes; reduce the step size for a finer grid.

In [ ]:
step = 20   # degrees; smaller = finer grid = significantly longer run time
phi1_vals = np.arange(0, 360 + step, step)
Phi_vals = np.arange(0, 180 + step, step)
phi2_vals = np.arange(0, 360 + step, step)

X_list, Y_list, Z_list, E_list = [], [], [], []
for phi1 in phi1_vals:
    for Phi_ in Phi_vals:
        for phi2 in phi2_vals:
            rot = Rotation.from_euler('ZXZ', [phi1, Phi_, phi2], degrees=True)
            try:
                E_val = calculate_Eeff(rot.as_matrix(), C1, interface.A2, interface.B2)
            except Exception:
                continue  # skip degenerate orientation
            v = rot.apply([0, 0, 1])
            X_list.append(v[0] * E_val)
            Y_list.append(v[1] * E_val)
            Z_list.append(v[2] * E_val)
            E_list.append(E_val)

X_arr, Y_arr, Z_arr, E_arr = map(np.array, (X_list, Y_list, Z_list, E_list))

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
norm = plt.Normalize(E_arr.min(), E_arr.max())
colors = plt.cm.viridis(norm(E_arr))
ax.scatter(X_arr, Y_arr, Z_arr, c=colors, s=40)
mappable = plt.cm.ScalarMappable(cmap='viridis', norm=norm)
mappable.set_array(E_arr)
fig.colorbar(mappable, ax=ax, shrink=0.5, aspect=10, label='4/E22 (effective stiffness)')
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_box_aspect([1, 1, 1])
ax.set_title("Effective interface stiffness versus misorientation of grain 1")
plt.tight_layout()
plt.show()

## 10. Appendix: tools for crystal orientation

Helper functions to determine Euler angles from a crystal direction (e.g. `[1,1,1]`) and
vice versa, and to convert to Miller indices. They are not needed by the sections above,
but are useful when setting up new grain orientations.

In [ ]:
uvw = np.array([1., 1., 1.])
ref = np.array([0., 0., 1.])
phi1, Phi, phi2 = EulerAnglesDegFromDirection(uvw, ref)
print(f"Euler angles for [1,1,1]: phi1={phi1:.2f}, Phi={Phi:.2f}, phi2={phi2:.2f}")
v_check = direction_from_EulerAnglesDeg(phi1, Phi, phi2, ref)
print("Back-computed direction:", np.round(v_check, 6), " Miller:", vector_to_miller(v_check))

## 11. Validation

### 11.1 Closed-form stresses versus finite differences

The closed-form stresses are compared with the stresses obtained from central finite
differences of the displacement field (`strain_stress_fd`: strains from $\nabla\mathbf u$,
$\varepsilon_{33}=0$, Hooke's law), an independent numerical reference. The points lie
away from the crack tip and from the interface line, where finite differences are not
applicable. Expected: relative deviation below $10^{-6}$.

In [ ]:
pts = [(-0.8 * x_K, 0.5 * x_K), (0.6 * x_K, 0.3 * x_K), (-0.4 * x_K, 0.9 * x_K)]

def max_rel_dev(u_fun, s_fun, C, points):
    worst = 0.0
    for (px, py) in points:
        s_fd = strain_stress_fd(u_fun, px, py, C, a)
        s_an = np.array(s_fun(px, py))
        worst = max(worst, np.abs(s_an - s_fd).max() / np.abs(s_fd).max())
    return worst

dev_single = max_rel_dev(
    lambda x, y: single.displacement(x, y, K_I, K_II, K_III, a),
    lambda x, y: single.stress(x, y, K_I, K_II, K_III, a), single.C_rot, pts + [(px, -py) for px, py in pts])
dev_top = max_rel_dev(
    lambda x, y: interface.displacement(x, y, K_I, K_II, K_III, a, side="top"),
    lambda x, y: interface.stress(x, y, K_I, K_II, K_III, a, side="top"), interface.C_rot_top, pts)
dev_bot = max_rel_dev(
    lambda x, y: interface.displacement(x, y, K_I, K_II, K_III, a, side="bot"),
    lambda x, y: interface.stress(x, y, K_I, K_II, K_III, a, side="bot"), interface.C_rot_bot,
    [(px, -py) for px, py in pts])

print(f"single crystal   : max. relative deviation = {dev_single:.2e}")
print(f"interface, top   : max. relative deviation = {dev_top:.2e}")
print(f"interface, bottom: max. relative deviation = {dev_bot:.2e}")
assert max(dev_single, dev_top, dev_bot) < 1e-6

The same check for grain pairs with a general misorientation (non-zero $\hat{\mathbf S}$),
together with the comparison against the interface solution of Hwu (2010, Eqs. 7.112/7.113),
the continuity of $\mathbf u$ and $\sigma_{i2}$ across the bonded interface and the
equilibrium condition, is performed by `validate_stress_interface.py`.

### 11.2 Isotropic solution versus the Stroh solution

In the isotropic limit the anisotropic (Stroh) solution must reproduce the Williams
solution. Exactly isotropic constants give repeated Stroh eigenvalues, so a slightly perturbed
stiffness ($C_{44}$ increased by a small shift) is used; the deviation must vanish
proportionally to the shift. All three modes and a mixed load are checked on points around the
tip (plane strain, i.e. $\sigma_{33}=\nu(\sigma_{11}+\sigma_{22})$).

In [ ]:
K0 = 5 * np.sqrt(1000.)
pts_iso = [(r * np.cos(t), r * np.sin(t)) for r in np.geomspace(2e-4, 5e-3, 6)
           for t in np.linspace(-170, 170, 13) * np.pi / 180]
iso_loads = [(K0, 0., 0.), (0., K0, 0.), (0., 0., K0), (.5 * K0, .3 * K0, .8 * K0)]
ref_iso = IsotropicCrackModel("Steel (isotropic)")

previous = None
for shift in (100., 10., 1.):
    name = f"Steel (isotropic, C44 shift {shift:g} MPa)"
    MATERIALS[name] = {"C": isotropic_stiffness(E=210000., nu=0.3, degeneracy_shift=shift),
                       "symmetry": "isotropic (perturbed)", "note": "check only"}
    stroh = AnisotropicCrackModel(name, 0., 0., 0.)
    dev_u = dev_s = 0.0
    for K in iso_loads:
        U_ref = np.array([ref_iso.displacement(x, y, *K) for x, y in pts_iso])
        U_str = np.array([stroh.displacement(x, y, *K, a) for x, y in pts_iso])
        S_ref = np.array([ref_iso.stress(x, y, *K) for x, y in pts_iso])
        S_str = np.array([stroh.stress(x, y, *K, a) for x, y in pts_iso])
        dev_u = max(dev_u, np.abs(U_str - U_ref).max() / np.abs(U_ref).max())
        dev_s = max(dev_s, np.abs(S_str - S_ref).max() / np.abs(S_ref).max())
    print(f"C44 shift {shift:6g} MPa:  max. rel. deviation  u = {dev_u:.2e}   sigma = {dev_s:.2e}")
    del MATERIALS[name]
    if previous:
        assert dev_u < previous[0] and dev_s < previous[1]       # decreases with the shift
    previous = (dev_u, dev_s)
assert dev_u < 1e-4 and dev_s < 1e-5

### 11.3 Reference values of the original notebook

The following values were computed by the original (Colab) implementation for the
default configuration of section 3 (Fe, $\varphi_1=27^\circ$ above Fe, $\varphi_1=0^\circ$
below, $K_I=5\sqrt{1000}$ MPa$\sqrt{\text{mm}}$, $a=0.05$ mm).

In [ ]:
default_config = (
    np.isclose(K_I, 5 * np.sqrt(1000.)) and K_II == 0 and K_III == 0 and a == 0.05
    and material_1_name == material_2_name == "Fe (cubic, example)"
    and (phi1_1, Phi_1, phi2_1) == (27., 0., 0.) and (phi1_2, Phi_2, phi2_2) == (0., 0., 0.)
)

if not default_config:
    print("Reference values only apply to the default configuration of section 3 - skipped.")
else:
    R27 = Rotation.from_euler('ZXZ', [27., 0., 0.], degrees=True).as_matrix()
    checks = [
        ("u1(-0.01, 1e-10) [mm]", u1p, 5.90774034279533e-13, 1e-6),
        ("u2(-0.01, 1e-10) [mm]", u2p, 1.18154792377224e-4, 1e-8),
        ("E22 [1/MPa]", E22, 1.873144455886489e-05, 1e-12),
        ("Zener factor A", zener_A, 2.359735638027453, 1e-12),
        ("4/E22 for 27 deg vs 0 deg [MPa]", calculate_Eeff(R27, C1, interface.A2, interface.B2), 213544.662, 1e-8),
    ]
    for label, value, ref_value, tol in checks:
        rel = abs(value - ref_value) / abs(ref_value)
        print(f"{label:34s} new = {value:.12e}   reference = {ref_value:.12e}   rel. dev. = {rel:.1e}   {'OK' if rel < tol else 'DEVIATION'}")
        assert rel < tol